In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [0]:
catalog_name = "ecommerce"


## Products

In [0]:
df_products = spark.table(f"{catalog_name}.silver.slv_products")

df_brands = spark.table(f"{catalog_name}.silver.slv_brands")

df_category = spark.table(f"{catalog_name}.silver.slv_category")

In [0]:
df_products.createOrReplaceTempView("v_products")

df_brands.createOrReplaceTempView("v_brands")

df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("select * from v_products limit 5"))

In [0]:
display(spark.sql("select * from v_category limit 5"))

In [0]:
display(spark.sql("select * from v_brands limit 5"))

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%sql
create or replace table gold.gld_dim_products as
with brands_categories as (
    select
        b.brand_name,
        b.brand_code,
        c.category_name,
        c.category_code
    from v_brands b
    inner join v_category c
    on
        b.category_code = c.category_code
)
select
    p.product_id,
    p.sku,
    p.category_code,
    coalesce(bc.category_name,'Not Available') as category_name,
    coalesce(bc.brand_name,'Not Available') as brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.file_name,
    p.ingest_timestamp
from v_products p
left join brands_categories bc
    on p.brand_code = bc.brand_code;

## Customers

In [0]:
india_region = {
    "MH":"West","Gj" : "West","Rj":"West",
    "KA":"South","TN": "South","TS":"South","AP":"South","KL":"South",
    "UP":"North","WB":"North","DL":"North"
}

australia_region ={
    "VIC":"SouthEast","WA":"West","NSW":"East","QLD":"NorthEast"
}

uk_region ={
    "ENG":"England","WSL":"Wales","NIR":"Northern Ireland","SCT":"Scotland"
}

us_region = {
    "MA":"NorthEast","FL":"South","NJ":"NorthEasr","CA":"West",
    "NY":"NorthEast","TX":"South"
}

uae_region ={
    "AUH":"Abu Dhabi","DU":"Dubai","SHJ":"Sharjah"
}

singapore_region ={
    "SG":"Singapore"
}

canada_region = {
    "BC":"West","AB":"West","ON":"East","QC":"East","NS":"East","IL":"Other"
}

country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates":uae_region,
    "Singapore":singapore_region,
    "Canada":canada_region
}

In [0]:
country_state_map

In [0]:
rows = []
for country, states in country_state_map.items():
    for state_code,region in states.items():
        rows.append(Row(country=country,state=state_code,region=region))
rows[:10]

In [0]:
df_region_mapping = spark.createDataFrame(rows)

df_region_mapping.show(truncate=False)

In [0]:
df_silver = spark.table(f'{catalog_name}.silver.slv_customers')
display(df_silver.limit(5))

In [0]:
df_gold = df_silver.join(df_region_mapping, on=['country', 'state'], how='left')

df_gold = df_gold.fillna({'region': 'Other'})

display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

## Date/Calendar

In [0]:
df_silver = spark.table(f'{catalog_name}.silver.slv_calendar')
display(df_silver.limit(5))

In [0]:
df_gold = df_silver.withColumn("date_id",date_format(col("date"),"yyyyMMdd").cast("int"))

df_gold = df_gold.withColumn("month_name",date_format(col("date"),"MMMM"))

df_gold = df_gold.withColumn(
    "is_weekend",
    when(col("day_name").isin("Saturday","Sunday"),1).otherwise(0)
)

display(df_gold.limit(5))

In [0]:
desired_columns_order = ["date_id","date","year","month_name","day_name","is_weekend","quarter","week","_ingested_at","_source_file"]

df_gold = df_gold.select(desired_columns_order)

display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")

In [0]:
%sql
describe extended ecommerce.gold.gld_dim_date;